# Double Cheking for CUDA

In [ ]:
%cd /content

/content


In [ ]:
!nvcc --version
!nvidia-smi

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Tue Feb 24 10:33:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8       

# ROOT CMAKELISTS

In [ ]:
%%writefile CMakeLists.txt
cmake_minimum_required(VERSION 3.18)

project(Assignment02 LANGUAGES CXX CUDA)

# Target the Tesla T4 architecture
set(CMAKE_CUDA_ARCHITECTURES 75)

# Strict flags specifically for C++ files
add_compile_options("$<$<COMPILE_LANGUAGE:CXX>:-Wall;-Wextra;-Werror;-O3>")

# Strict flags specifically for CUDA files (passing warnings down via -Xcompiler)
add_compile_options("$<$<COMPILE_LANGUAGE:CUDA>:-Xcompiler=-Wall,-Wextra,-Werror;-O3;-lineinfo>")

add_subdirectory(task01)
add_subdirectory(matrix_io)
add_subdirectory(task02)
add_subdirectory(task03)

Overwriting CMakeLists.txt


In [ ]:
!mkdir build

mkdir: cannot create directory ‘build’: File exists


# TASK 1

In [ ]:
!mkdir task01

mkdir: cannot create directory ‘task01’: File exists


In [ ]:
%%writefile task01/CMakeLists.txt
add_executable(device_info device_info.cu)

Overwriting task01/CMakeLists.txt


In [ ]:
%%writefile task01/device_info.cu
#include <stdio.h>

int main() {
    int deviceId = 0;
    cudaDeviceProp devProp;
    cudaGetDeviceProperties(&devProp, deviceId);

    // Fetch clocks using attributes (Required for CUDA 13.0+)
    int clockRateKHz = 0;
    int memoryClockRateKHz = 0;
    cudaDeviceGetAttribute(&clockRateKHz, cudaDevAttrClockRate, deviceId);
    cudaDeviceGetAttribute(&memoryClockRateKHz, cudaDevAttrMemoryClockRate, deviceId);

    printf("--- 8 Important Properties (from cudaDeviceProp) ---\n");

    // 1. Device Name
    printf("devProp.name: represents the GPU Model. Value = %s\n", devProp.name);

    // 2. Global Memory
    printf("devProp.totalGlobalMem: represents the total VRAM in the GPU. Value = %zu bytes\n", devProp.totalGlobalMem);

    // 3. Streaming Multiprocessors (SMs)
    printf("devProp.multiProcessorCount: represents how many SMs (fundamental units of processing) are in the GPU. Value = %d\n", devProp.multiProcessorCount);

    // 4. Max Threads per Block
    printf("devProp.maxThreadsPerBlock: represents the maximum number of threads that can be launched in a single block. Value = %d\n", devProp.maxThreadsPerBlock);

    // 5. Warp Size
    printf("devProp.warpSize: represents the number of threads (32) grouped by the SM to execute instructions in lockstep. Value = %d\n", devProp.warpSize);

    // 6. Shared Memory per Block
    printf("devProp.sharedMemPerBlock: represents how much fast on-chip cache/SRAM is available per block. Value = %zu bytes\n", devProp.sharedMemPerBlock);

    // 7. Max Grid Size
    printf("devProp.maxGridSize[0]: represents the maximum number of blocks allowed in the X direction. Value = %d\n", devProp.maxGridSize[0]);

    // 8. Memory Bus Width
    printf("devProp.memoryBusWidth: represents how many bits can pass simultaneously across the memory bus. Value = %d bits\n", devProp.memoryBusWidth);


    printf("\n---Calculated Values---\n");

    // 9. Max Global Memory Bandwidth (GB/s)
    double memBandwidth = 2.0 * memoryClockRateKHz * (devProp.memoryBusWidth / 8.0) / 1000000.0;
    printf("Max Global Memory Bandwidth: represents how fast memory can be accessed in the global memory. Value = %.2f GB/s\n", memBandwidth);

    // 10. Peak Compute Performance (TFLOPs)
    int coresPerSM = 64; // Hardcoded for Tesla T4 (64 cores per SM)
    double peakTFLOPs = (2.0 * devProp.multiProcessorCount * coresPerSM * clockRateKHz) / 1e9;
    printf("Peak Compute Performance: represents how many trillions of operations can be performed by the GPU/s in ideal conditions. Value = %.2f TFLOPs\n", peakTFLOPs);




    return 0;
}

Overwriting task01/device_info.cu


# matrix I/O

In [ ]:
%mkdir matrix_io

mkdir: cannot create directory ‘matrix_io’: File exists


In [ ]:
!touch matrix_io/__init__.py

In [ ]:
%%writefile matrix_io/generate_data.py
import random
import sys

def generate_input_file(filename, M =3, K=4, N=2):
    with open(filename, 'w') as f:
        f.write(f"{M} {K}\n")
        for _ in range(M):
            row = [f"{random.uniform(-1000.0, 1000.0):.2f}" for _ in range(K)]
            f.write(" ".join(row) + "\n")

        f.write(f"{K} {N}\n")
        for _ in range(K):
            row = [f"{random.uniform(-1000.0, 1000.0):.2f}" for _ in range(N)]
            f.write(" ".join(row) + "\n")

    print(f"Successfully generated {filename} with A({M}x{K}) and B({K}x{N})")

Overwriting matrix_io/generate_data.py


In [ ]:
%%writefile matrix_io/matrix_io.hpp

#ifndef MATRIX_IO_HPP
#define MATRIX_IO_HPP

#include <vector>
#include <iostream>

// Struct to hold matrix data in a contiguous 1D array
struct Matrix {
    int rows;
    int cols;
    std::vector<float> data;
};

// Function prototypes
Matrix readMatrix(std::istream& is);
void writeMatrix(std::ostream& os, const Matrix& mat);

#endif

Overwriting matrix_io/matrix_io.hpp


In [ ]:
%%writefile matrix_io/matrix_io.cpp
#include "matrix_io.hpp"
#include <stdexcept>
#include <string>

// Reads a single matrix block from an input stream
Matrix readMatrix(std::istream& is) {
    Matrix mat;
    if (!(is >> mat.rows >> mat.cols)) {
        throw std::runtime_error("Failed to read matrix dimensions or EOF reached.");
    }

    mat.data.resize(mat.rows * mat.cols);
    for (int i = 0; i < mat.rows * mat.cols; ++i) {
        if (!(is >> mat.data[i])) {
            throw std::runtime_error("Failed to read matrix data elements.");
        }
    }
    return mat;
}

// Writes a single matrix block to an output stream
void writeMatrix(std::ostream& os, const Matrix& mat) {
    os << mat.rows << " " << mat.cols << "\n";
    for (int i = 0; i < mat.rows; ++i) {
        for (int j = 0; j < mat.cols; ++j) {
            os << mat.data[i * mat.cols + j];
            if (j < mat.cols - 1) os << " ";
        }
        os << "\n";
    }
}

Overwriting matrix_io/matrix_io.cpp


In [ ]:
%%writefile matrix_io/CMakeLists.txt
add_library(matrix_io_lib matrix_io.cpp)

target_include_directories(matrix_io_lib PUBLIC ${CMAKE_CURRENT_SOURCE_DIR})

Overwriting matrix_io/CMakeLists.txt


# TASK 2

In [ ]:
%mkdir task02

mkdir: cannot create directory ‘task02’: File exists


In [ ]:
%%writefile task02/CMakeLists.txt
add_executable(task02_app main.cpp)
target_link_libraries(task02_app PRIVATE matrix_io_lib)

Overwriting task02/CMakeLists.txt


In [ ]:
%%writefile task02/main.cpp
#include <chrono>
#include <fstream>
#include <iostream>
#include <stdexcept>
#include <string>
#include <vector>
#include "matrix_io.hpp"

Matrix multiplyMatricesCPU(const Matrix& A, const Matrix& B) {
    if (A.cols != B.rows) {
        throw std::invalid_argument("Matrix dimensions do not match for multiplication.");
    }

    Matrix C;
    C.rows = A.rows;
    C.cols = B.cols;
    C.data.assign(C.rows * C.cols, 0.0);

    for (int i = 0; i < A.rows; ++i) {
        for (int j = 0; j < B.cols; ++j) {
            float sum = 0.0;
            for (int k = 0; k < A.cols; ++k) {
                // index = row * width + col
                sum += A.data[i * A.cols + k] * B.data[k * B.cols + j];
            }
            C.data[i * C.cols + j] = sum;
        }
    }

    return C;
}

int main(int argc, char* argv[]) {
    if (argc < 2 || argc > 3) {
        std::cerr << "Usage: " << argv[0] << " <input_file> [output_file]\n";
        return 1;
    }

    std::string inputFile = argv[1];
    std::string outputFile = (argc == 3) ? argv[2] : "";

    try {
        std::ifstream inFile(inputFile);
        if (!inFile.is_open()) {
            throw std::runtime_error("Could not open input file: " + inputFile);
        }
        Matrix A = readMatrix(inFile);
        Matrix B = readMatrix(inFile);
        inFile.close();

        // start timer
        auto start = std::chrono::high_resolution_clock::now();

        Matrix C = multiplyMatricesCPU(A, B);

        // stop timer
        auto end = std::chrono::high_resolution_clock::now();
        std::chrono::duration<double, std::milli> duration = end - start;

        if (!outputFile.empty()) {
            std::ofstream outFile(outputFile);
            if (!outFile.is_open()) {
                throw std::runtime_error("Could not open output file: " + outputFile);
            }
            writeMatrix(outFile, C);
            outFile.close();
        } else {
            writeMatrix(std::cout, C);
        }

        // print duration
        std::cout << duration.count() << "\n";

    } catch (const std::exception& e) {
        std::cerr << "Error: " << e.what() << "\n";
        return 1;
    }

    return 0;
}

Overwriting task02/main.cpp


# TASK 3

In [ ]:
%mkdir -p task03

In [ ]:
%%writefile task03/CMakeLists.txt
add_executable(task03_app main.cpp gpu_wrapper.cu)

target_link_libraries(task03_app PRIVATE matrix_io_lib)

Overwriting task03/CMakeLists.txt


In [ ]:
%%writefile task03/gpu_wrapper.hpp
#ifndef GPU_WRAPPER_HPP
#define GPU_WRAPPER_HPP

#include "matrix_io.hpp"

Matrix multiplyMatricesGPU(const Matrix& A, const Matrix& B, bool useTiling = false);

#endif

Overwriting task03/gpu_wrapper.hpp


In [ ]:
%%writefile task03/gpu_wrapper.cu
#include "gpu_wrapper.hpp"
#include <iostream>
#include <stdexcept>

#define TILE_SIZE 16

#define cudaCheckError(ans) { gpuAssert((ans), __FILE__, __LINE__); }
inline void gpuAssert(cudaError_t code, const char *file, int line, bool abort=true) {
    if (code != cudaSuccess) {
        fprintf(stderr,"GPUassert: %s %s %d\n", cudaGetErrorString(code), file, line);
        if (abort) exit(code);
    }
}

// --- KERNEL 1: NAIVE ---
__global__ void matrixMulNaiveKernel(const float* A, const float* B, float* C, int M, int K, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < M && col < N) {
        float sum = 0.0;
        for (int i = 0; i < K; ++i) {
            sum += A[row * K + i] * B[i * N + col];
        }
        C[row * N + col] = sum;
    }
}

// --- KERNEL 2: TILED ---
__global__ void matrixMulTiledKernel(const float* A, const float* B, float* C, int M, int K, int N) {
    __shared__ float tile_A[TILE_SIZE][TILE_SIZE+1];
    __shared__ float tile_B[TILE_SIZE][TILE_SIZE+1];

    int bx = blockIdx.x; int by = blockIdx.y;
    int tx = threadIdx.x; int ty = threadIdx.y;

    int row = by * TILE_SIZE + ty;
    int col = bx * TILE_SIZE + tx;

    float sum = 0.0;

    for (int t = 0; t < (K + TILE_SIZE - 1) / TILE_SIZE; ++t) {
        if (row < M && (t * TILE_SIZE + tx) < K) {
            tile_A[ty][tx] = A[row * K + t * TILE_SIZE + tx];
        } else {
            tile_A[ty][tx] = 0.0;
        }
        if ((t * TILE_SIZE + ty) < K && col < N) {
            tile_B[ty][tx] = B[(t * TILE_SIZE + ty) * N + col];
        } else {
            tile_B[ty][tx] = 0.0;
        }

        __syncthreads();
        #pragma unroll
        for (int i = 0; i < TILE_SIZE; ++i) {
            sum += tile_A[ty][i] * tile_B[i][tx];
        }

        __syncthreads();
    }

    if (row < M && col < N) {
        C[row * N + col] = sum;
    }
}

Matrix multiplyMatricesGPU(const Matrix& A, const Matrix& B, bool useTiling) {
    if (A.cols != B.rows) throw std::invalid_argument("Matrix dimensions mismatch!");

    int M = A.rows; int K = A.cols; int N = B.cols;
    Matrix C; C.rows = M; C.cols = N; C.data.resize(M * N, 0.0);

    float *d_A, *d_B, *d_C;
    size_t bytesA = M * K * sizeof(float);
    size_t bytesB = K * N * sizeof(float);
    size_t bytesC = M * N * sizeof(float);

    cudaCheckError(cudaMalloc(&d_A, bytesA));
    cudaCheckError(cudaMalloc(&d_B, bytesB));
    cudaCheckError(cudaMalloc(&d_C, bytesC));

    cudaCheckError(cudaMemcpy(d_A, A.data.data(), bytesA, cudaMemcpyHostToDevice));
    cudaCheckError(cudaMemcpy(d_B, B.data.data(), bytesB, cudaMemcpyHostToDevice));

    // Dynamic Kernel Launch
    if (useTiling) {
        dim3 threads(TILE_SIZE, TILE_SIZE);
        dim3 blocks((N + TILE_SIZE - 1) / TILE_SIZE, (M + TILE_SIZE - 1) / TILE_SIZE);
        matrixMulTiledKernel<<<blocks, threads>>>(d_A, d_B, d_C, M, K, N);
    } else {
        dim3 threads(16, 16);
        dim3 blocks((N + 15) / 16, (M + 15) / 16);
        matrixMulNaiveKernel<<<blocks, threads>>>(d_A, d_B, d_C, M, K, N);
    }

    cudaCheckError(cudaPeekAtLastError());
    cudaCheckError(cudaDeviceSynchronize());

    cudaCheckError(cudaMemcpy(C.data.data(), d_C, bytesC, cudaMemcpyDeviceToHost));

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);

    return C;
}

Overwriting task03/gpu_wrapper.cu


In [ ]:
%%writefile task03/main.cpp
#include <chrono>
#include <fstream>
#include <iostream>
#include <string>
#include <stdexcept>

#include "gpu_wrapper.hpp"
#include "matrix_io.hpp"

int main(int argc, char* argv[]) {
    if (argc < 2 || argc > 4) {
        std::cerr << "Usage: " << argv[0] << " <input_file> [output_file] [--tiled]\n";
        return 1;
    }

    std::string inputFile = "";
    std::string outputFile = "";
    bool useTiling = false;

    for (int i = 1; i < argc; ++i) {
        std::string arg = argv[i];
        if (arg == "--tiled") {
            useTiling = true;
        } else if (inputFile.empty()) {
            inputFile = arg;
        } else if (outputFile.empty()) {
            outputFile = arg;
        }
    }

    try {
        std::ifstream inFile(inputFile);
        if (!inFile.is_open()) {
            throw std::runtime_error("Could not open input file: " + inputFile);
        }

        Matrix A = readMatrix(inFile);
        Matrix B = readMatrix(inFile);
        inFile.close();
        // start timer
        auto start = std::chrono::high_resolution_clock::now();

        Matrix C = multiplyMatricesGPU(A, B, useTiling);
        // stop timer

        auto end = std::chrono::high_resolution_clock::now();
        std::chrono::duration<double, std::milli> duration = end - start;

        if (!outputFile.empty()) {
            std::ofstream outFile(outputFile);
            if (!outFile.is_open()) {
                throw std::runtime_error("Could not open output file: " + outputFile);
            }
            writeMatrix(outFile, C);
            outFile.close();
        } else {
            writeMatrix(std::cout, C);
        }
        // print duration
        std::cout << duration.count() << "\n";

    } catch (const std::exception& e) {
        std::cerr << "Error: " << e.what() << "\n";
        return 1;
    }

    return 0;
}

Overwriting task03/main.cpp


# TASK 4

In [ ]:
%%writefile benchmark.py

import matplotlib.pyplot as plt
import os
import subprocess
import sys

sys.path.append(os.path.abspath("."))
from matrix_io.generate_data import generate_input_file

dimensions = [
    (10, 10, 10),
    # gpu is now warmed up
    (512, 256, 512),
    (128, 4096, 128),
    (768, 512, 1024),
    (1024, 768, 1536),
    (4096,128,2048),
    (128,38000,128)
]

cpu_times = []
gpu_times = []
labels = []

for M, K, N in dimensions:
    label = f"({M}x{K}) * ({K}x{N})"
    labels.append(label)

    print(f"\n--- Benchmarking Dimensions: {label} ---")
    generate_input_file("bench_input.txt", M, K, N)

    # Run CPU executable
    cpu_result = subprocess.run(["./build/task02/task02_app", "bench_input.txt", "output_cpu.txt"], capture_output=True, text=True)
    try:
        cpu_time = float(cpu_result.stdout.strip())
        cpu_times.append(cpu_time)
    except ValueError:
        print("Error parsing CPU time. Output was:", cpu_result.stdout)
        cpu_times.append(0)

    # Run GPU executable
    gpu_result = subprocess.run(["./build/task03/task03_app", "bench_input.txt", "output_gpu.txt"], capture_output=True, text=True)
    try:
        gpu_time = float(gpu_result.stdout.strip())
        gpu_times.append(gpu_time)
    except ValueError:
        print("Error parsing GPU time. Output was:", gpu_result.stdout)
        gpu_times.append(0)


# Plotting the results
plt.figure(figsize=(12, 6))
plt.plot(labels, cpu_times, label='CPU Time (ms)', marker='o', color='red')
plt.plot(labels, gpu_times, label='GPU Time (ms) [inc. Memory Transfer]', marker='s', color='blue')
plt.title("Matrix Multiplication: CPU vs GPU (Non-Square Matrices)")
plt.xlabel("Matrix Dimensions (M x K * K x N)")
plt.ylabel("Execution Time (milliseconds)")
plt.xticks(rotation=10)
plt.tight_layout()
plt.legend()
plt.grid(True)
plt.savefig("task04_benchmark_plot.png")
print("\nBenchmarking complete! Plot saved as 'task04_benchmark_plot.png'")

Overwriting benchmark.py


# Task 5

In [ ]:
# %%writefile benchmark.py

# import matplotlib.pyplot as plt
# import os
# import subprocess
# import sys

# sys.path.append(os.path.abspath("."))
# from matrix_io.generate_data import generate_input_file

# # --- CONFIGURATION ---
# small_dimensions = [
#     (10, 10, 10),           # Warmup
#     (8192, 128, 128),       # ~134 Million ops (Large M)
#     (128, 512, 8192),       # ~536 Million ops (Large N)
#     (16384, 256, 512),      # ~2.1 Billion ops (Huge M)
#     (512, 1024, 8192),      # ~4.2 Billion ops (Large N)
#     (2048, 8192, 512)       # ~8.5 Billion ops (Large K)
# ]

# # Phase 2: GPU Naive vs GPU Tiled (Up to ~1.1 Trillion Ops)
# large_dimensions = [
#     (32768, 128, 4096),     # ~17 Billion ops (Extreme M)
#     (4096, 512, 32768),     # ~68 Billion ops (Extreme N)
#     (1024, 16384, 8192),    # ~137 Billion ops (Extreme K)
#     (8192, 2048, 16384),    # ~274 Billion ops
#     (8192, 8192, 4096)      # ~274 Billion ops
# ]

# def run_benchmark(executable_args):
#     result = subprocess.run(executable_args, capture_output=True, text=True)
#     if result.returncode != 0:
#         print(f"Error running {' '.join(executable_args)}:\n{result.stderr.strip()}")
#         return 0.0
#     try:
#         return float(result.stdout.strip())
#     except ValueError:
#         print(f"Error parsing time. Output was: {result.stdout}")
#         return 0.0

# print("=======================================")
# print("PHASE 1: CPU vs GPU vs GPU (Tiled)")
# print("=======================================")

# labels_small, cpu_times, gpu_naive_times_small, gpu_tiled_times_small = [], [], [], []

# for M, K, N in small_dimensions:
#     label = f"{M}x{K}x{N}"
#     labels_small.append(label)
#     print(f"\n--- Dimensions: {label} ---")
#     generate_input_file("bench_input.txt", M, K, N)

#     # 1. CPU
#     cpu_times.append(run_benchmark(["./build/task02/task02_app", "bench_input.txt", "output_cpu.txt"]))
#     # 2. GPU Naive
#     gpu_naive_times_small.append(run_benchmark(["./build/task03/task03_app", "bench_input.txt", "output_gpu.txt"]))
#     # 3. GPU Tiled
#     gpu_tiled_times_small.append(run_benchmark(["./build/task03/task03_app", "bench_input.txt", "output_gpu.txt", "--tiled"]))

# # Plot Phase 1
# plt.figure(figsize=(10, 6))
# plt.plot(labels_small[1:], cpu_times[1:], label='CPU Time', marker='o', color='red') # Skip warmup index 0
# plt.plot(labels_small[1:], gpu_naive_times_small[1:], label='GPU (Naive)', marker='s', color='blue')
# plt.plot(labels_small[1:], gpu_tiled_times_small[1:], label='GPU (Tiled Shared Mem)', marker='^', color='green')
# plt.title("Phase 1: CPU vs GPU Performance (Up to ~8.5B Ops)")
# plt.xlabel("Matrix Dimensions (M x K x N)")
# plt.ylabel("Execution Time (ms)")
# plt.xticks(rotation=10)
# plt.tight_layout()
# plt.legend()
# plt.grid(True)
# plt.savefig("task05_phase1_plot.png")
# print("\nPhase 1 Plot saved as 'task05_phase1_plot.png'")


# print("\n=======================================")
# print("PHASE 2: GPU Naive vs GPU (Tiled)")
# print("=======================================")

# labels_large, gpu_naive_times_large, gpu_tiled_times_large = [], [], []

# for M, K, N in large_dimensions:
#     label = f"{M}x{K}x{N}"
#     labels_large.append(label)
#     print(f"\n--- Dimensions: {label} ---")
#     generate_input_file("bench_input.txt", M, K, N)

#     # 1. GPU Naive
#     gpu_naive_times_large.append(run_benchmark(["./build/task03/task03_app", "bench_input.txt", "output_gpu.txt"]))
#     # 2. GPU Tiled
#     gpu_tiled_times_large.append(run_benchmark(["./build/task03/task03_app", "bench_input.txt", "output_gpu.txt", "--tiled"]))

# # Plot Phase 2
# plt.figure(figsize=(10, 6))
# plt.plot(labels_large, gpu_naive_times_large, label='GPU (Naive Global Mem)', marker='s', color='blue')
# plt.plot(labels_large, gpu_tiled_times_large, label='GPU (Tiled Shared Mem)', marker='^', color='green')
# plt.title("Phase 2: GPU Naive vs Tiled Performance (Up to 1.1 Trillion Ops)")
# plt.xlabel("Matrix Dimensions (M x K x N)")
# plt.ylabel("Execution Time (ms)")
# plt.xticks(rotation=10)
# plt.tight_layout()
# plt.legend()
# plt.grid(True)
# plt.savefig("task05_phase2_plot.png")
# print("\nPhase 2 Plot saved as 'task05_phase2_plot.png'")

In [ ]:
!cd build && cmake .. && make

-- Configuring done (0.0s)
-- Generating done (0.0s)
-- Build files have been written to: /content/build
[ 11%] Building CUDA object task01/CMakeFiles/device_info.dir/device_info.cu.o
[ 22%] Linking CUDA executable device_info
[ 22%] Built target device_info
[ 33%] Building CXX object matrix_io/CMakeFiles/matrix_io_lib.dir/matrix_io.cpp.o
[ 44%] Linking CXX static library libmatrix_io_lib.a
[ 44%] Built target matrix_io_lib
[ 55%] Building CXX object task02/CMakeFiles/task02_app.dir/main.cpp.o
[ 66%] Linking CXX executable task02_app
[ 66%] Built target task02_app
[ 77%] Building CXX object task03/CMakeFiles/task03_app.dir/main.cpp.o
[ 88%] Building CUDA object task03/CMakeFiles/task03_app.dir/gpu_wrapper.cu.o
[100%] Linking CXX executable task03_app
[100%] Built target task03_app


In [ ]:
!python benchmark.py


--- Benchmarking Dimensions: (10x10) * (10x10) ---
Successfully generated bench_input.txt with A(10x10) and B(10x10)

--- Benchmarking Dimensions: (512x256) * (256x512) ---
Successfully generated bench_input.txt with A(512x256) and B(256x512)

--- Benchmarking Dimensions: (128x4096) * (4096x128) ---
Successfully generated bench_input.txt with A(128x4096) and B(4096x128)

--- Benchmarking Dimensions: (768x512) * (512x1024) ---
Successfully generated bench_input.txt with A(768x512) and B(512x1024)

--- Benchmarking Dimensions: (1024x768) * (768x1536) ---
Successfully generated bench_input.txt with A(1024x768) and B(768x1536)

--- Benchmarking Dimensions: (4096x128) * (128x2048) ---
Successfully generated bench_input.txt with A(4096x128) and B(128x2048)

--- Benchmarking Dimensions: (128x38000) * (38000x128) ---
Successfully generated bench_input.txt with A(128x38000) and B(38000x128)

Benchmarking complete! Plot saved as 'task04_benchmark_plot.png'


In [ ]:
!ls

bench_input.txt  CMakeLists.txt  output_gpu.txt  task02
benchmark.py	 matrix_io	 sample_data	 task03
build		 output_cpu.txt  task01		 task04_benchmark_plot.png


# archive the directory for download

In [ ]:
!rm bench_input.txt output_cpu.txt output_gpu.txt

In [ ]:
%cd /

/


In [ ]:
import shutil

folder_to_zip = "content"
output_zip = "content_archive"

shutil.make_archive(output_zip, 'zip', folder_to_zip)

'/content_archive.zip'